# **Billing Data Cleaning**

## 1. Importing Libraries and Loading Data

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

import sys
sys.path.append(str(Path.cwd().parent))

from scripts.paths import RAW_DIR, CLEANED_DIR

df = pd.read_csv(RAW_DIR/"06_billing_dirty.csv")

## 2. Initial Data Inspection

In [4]:
df.head()

,Bill_ID,Encounter_ID,Patient_ID,Bill_Date,Bill_Category,Gross_Amount,Discount,Insurance_Amount,Tax,Net_Amount,Payment_Status,Payment_Mode
0,BL300000,E204388,P105967,2025-01-04,Consultation,54417.07,5095.34,24923.80,4476.53,28874.46,Pending,Bank Transfer
1,BL300001,E213154,P104582,2026-02-22,Consultation,227397.18,54047.44,12739.43,12758.92,173369.23,Paid,cash
2,BL300002,E208864,P106004,2026-08-14,Nursing,15397.64,2817.74,2611.92,102.93,10070.91,PENDING,NaN
3,BL300003,E209084,P100369,2026-01-10,Lab Test,178763.37,26754.55,96813.87,19703.52,74898.47,paid,cash
4,BL300004,E202692,P102804,2026-06-16,Consultation,99240.98,3870.86,36727.11,11674.66,70317.67,Cancelled,Card


In [5]:
df.shape

(12000, 12)

In [6]:
df.columns.tolist()

['Bill_ID',
 'Encounter_ID',
 'Patient_ID',
 'Bill_Date',
 'Bill_Category',
 'Gross_Amount',
 'Discount',
 'Insurance_Amount',
 'Tax',
 'Net_Amount',
 'Payment_Status',
 'Payment_Mode']

In [8]:
df.dtypes

Bill_ID              object
Encounter_ID         object
Patient_ID           object
Bill_Date            object
Bill_Category        object
Gross_Amount         object
Discount            float64
Insurance_Amount    float64
Tax                 float64
Net_Amount          float64
Payment_Status       object
Payment_Mode         object
dtype: object

In [9]:
df.isna().sum()

Bill_ID                0
Encounter_ID           0
Patient_ID             0
Bill_Date              0
Bill_Category          0
Gross_Amount          31
Discount               0
Insurance_Amount       0
Tax                    0
Net_Amount            36
Payment_Status      1742
Payment_Mode        1706
dtype: int64

In [10]:
df.duplicated().sum()

0

## 3. Audit of Specific Columns

In [11]:
df["Bill_ID"].duplicated().sum()

0

In [12]:
df["Encounter_ID"].duplicated().sum()

3733

In [13]:
df["Patient_ID"].duplicated().sum()

5771

In [14]:
df["Bill_Category"].value_counts(dropna=False)

Bill_Category
Consultation     1257
Surgery          1244
Medicine         1236
Misc             1228
Emergency Fee    1193
Lab Test         1192
Radiology        1178
Nursing          1171
Room Charges     1164
Procedure        1137
Name: count, dtype: int64

In [15]:
df["Payment_Status"].value_counts(dropna=False)

Payment_Status
PENDING      1761
NaN          1742
Paid         1728
Cancelled    1699
Partial      1697
Pending      1689
paid         1684
Name: count, dtype: int64

In [16]:
df["Payment_Mode"].value_counts(dropna=False)

Payment_Mode
UPI              1750
Card             1743
Cash             1723
Insurance        1722
NaN              1706
cash             1703
Bank Transfer    1653
Name: count, dtype: int64

In [17]:
bill_date_temp = pd.to_datetime(
    df["Bill_Date"],
    errors="coerce"
)

bill_date_temp.isna().sum()

0

In [18]:
bill_date_temp.min(), bill_date_temp.max()

(Timestamp('2025-01-01 00:00:00'), Timestamp('2026-08-24 00:00:00'))

In [20]:
gross_temp = df["Gross_Amount"].astype("string").str.replace(",", "", regex=False)

In [23]:
gross_numeric = pd.to_numeric(gross_temp, errors="coerce")

In [22]:
gross_numeric.isna().sum()

31

In [24]:
(gross_numeric <= 0).sum()

68

In [25]:
df.loc[gross_numeric <= 0, "Gross_Amount"].value_counts()

Gross_Amount
0       39
-500    29
Name: count, dtype: int64

In [26]:
df["Net_Amount"].isna().sum()

36

In [27]:
(df["Net_Amount"] <= 0).sum()

64

In [28]:
df.loc[
    df["Net_Amount"] <= 0,
    "Net_Amount"
].value_counts()

Net_Amount
 0.0       33
-1000.0    31
Name: count, dtype: int64

In [29]:
expected_net = (
    gross_numeric
    - df["Discount"]
    - df["Insurance_Amount"]
    + df["Tax"]
)

In [34]:
financial_mask = (
    gross_numeric.notna()
    & (gross_numeric > 0)
    & df["Net_Amount"].notna()
)

(df.loc[financial_mask, "Net_Amount"] - expected_net[financial_mask]).abs().gt(0.01).sum()

83

In [32]:
(gross_numeric.notna() & (gross_numeric > 0) & (expected_net < 0)).sum()

19

In [35]:
encounters_df = pd.read_csv(
    CLEANED_DIR / "05_encounters_cleaned.csv"
)

patients_df = pd.read_csv(
    CLEANED_DIR / "01_patients_cleaned.csv"
)

In [36]:
valid_encounter_ids = encounters_df["Encounter_ID"].dropna().unique()

df.loc[
    ~df["Encounter_ID"].isin(valid_encounter_ids),
    "Encounter_ID"
].value_counts(dropna=False)

Series([], Name: count, dtype: int64)

In [37]:
valid_patient_ids = patients_df["Patient_ID"].dropna().unique()

df.loc[
    ~df["Patient_ID"].isin(valid_patient_ids),
    "Patient_ID"
].value_counts(dropna=False)

Series([], Name: count, dtype: int64)

In [38]:
encounter_patient_map = (
    encounters_df
    .set_index("Encounter_ID")["Patient_ID"]
)

expected_patient = df["Encounter_ID"].map(encounter_patient_map)

(
    expected_patient.notna()
    & (df["Patient_ID"] != expected_patient)
).sum()

11997

## 4. Data Cleaning

### 4.1 Payment Status

In [39]:
payment_status_cleaned = (
    df["Payment_Status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "paid": "Paid",
        "pending": "Pending",
        "cancelled": "Cancelled",
        "partial": "Partial"
    })
)

In [40]:
payment_status_cleaned.value_counts(dropna=False)

Payment_Status
Pending      3450
Paid         3412
<NA>         1742
Cancelled    1699
Partial      1697
Name: count, dtype: Int64

In [41]:
df["Payment_Status"] = payment_status_cleaned

### 4.2 Payment Mode

In [42]:
payment_mode_cleaned = (
    df["Payment_Mode"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "cash": "Cash",
        "card": "Card",
        "upi": "UPI",
        "insurance": "Insurance",
        "bank transfer": "Bank Transfer"
    })
)

In [43]:
payment_mode_cleaned.value_counts(dropna=False)

Payment_Mode
Cash             3426
UPI              1750
Card             1743
Insurance        1722
<NA>             1706
Bank Transfer    1653
Name: count, dtype: Int64

In [44]:
df["Payment_Mode"] = payment_mode_cleaned

### 4.3 Bill Date

In [45]:
bill_date_cleaned = pd.to_datetime(
    df["Bill_Date"],
    errors="coerce"
)

df["Bill_Date"] = bill_date_cleaned

### 4.4 Gross Amount

In [46]:
gross_cleaned = pd.to_numeric(
    df["Gross_Amount"]
    .astype("string")
    .str.replace(",", "", regex=False),
    errors="coerce"
)

In [47]:
calculated_net = (
    gross_cleaned
    - df["Discount"]
    - df["Insurance_Amount"]
    + df["Tax"]
)

In [48]:
gross_cleaned = gross_cleaned.mask(
    gross_cleaned <= 0
)

gross_cleaned = gross_cleaned.mask(
    calculated_net < 0
)

In [49]:
df["Gross_Amount"] = gross_cleaned

### 4.5 Recalculate Net Amount

In [51]:
net_cleaned = (
    df["Gross_Amount"]
    - df["Discount"]
    - df["Insurance_Amount"]
    + df["Tax"]
)

In [52]:
df["Net_Amount"] = net_cleaned

## 5. Final Validation

In [53]:
df.shape

(12000, 12)

In [54]:
df.dtypes

Bill_ID                     object
Encounter_ID                object
Patient_ID                  object
Bill_Date           datetime64[ns]
Bill_Category               object
Gross_Amount               Float64
Discount                   float64
Insurance_Amount           float64
Tax                        float64
Net_Amount                 Float64
Payment_Status      string[python]
Payment_Mode        string[python]
dtype: object

In [55]:
df.isna().sum()

Bill_ID                0
Encounter_ID           0
Patient_ID             0
Bill_Date              0
Bill_Category          0
Gross_Amount         118
Discount               0
Insurance_Amount       0
Tax                    0
Net_Amount           118
Payment_Status      1742
Payment_Mode        1706
dtype: int64

In [56]:
df.duplicated().sum()

0

In [57]:
df["Bill_ID"].duplicated().sum()

0

In [58]:
df["Bill_Category"].unique()

array(['Consultation', 'Nursing', 'Lab Test', 'Procedure', 'Surgery',
       'Radiology', 'Emergency Fee', 'Misc', 'Medicine', 'Room Charges'],
      dtype=object)

In [60]:
df["Payment_Status"].unique()

<StringArray>
['Pending', 'Paid', 'Cancelled', 'Partial', <NA>]
Length: 5, dtype: string

In [61]:
df["Payment_Mode"].unique()

<StringArray>
['Bank Transfer', 'Cash', <NA>, 'Card', 'Insurance', 'UPI']
Length: 6, dtype: string

In [62]:
df["Gross_Amount"].min()

334.13

In [63]:
df["Net_Amount"].min()

74.83999999999995

In [64]:
expected_net_final = (
    df["Gross_Amount"]
    - df["Discount"]
    - df["Insurance_Amount"]
    + df["Tax"]
)

(
    df["Net_Amount"] - expected_net_final
).abs().gt(0.01).sum()

0

In [65]:
df["Bill_Date"].isna().sum()

0

In [66]:
df.loc[
    df["Encounter_ID"].notna() &
    ~df["Encounter_ID"].isin(valid_encounter_ids),
    "Encounter_ID"
]

Series([], Name: Encounter_ID, dtype: object)

In [67]:
df.loc[
    df["Patient_ID"].notna() &
    ~df["Patient_ID"].isin(valid_patient_ids),
    "Patient_ID"
]

Series([], Name: Patient_ID, dtype: object)

In [68]:
(expected_patient.notna() & (df["Patient_ID"] != expected_patient)).sum()

11997

## 6. Export Cleaned Data

In [69]:
df.to_csv(CLEANED_DIR / "06_billing_cleaned.csv", index=False)